#**Pipeline de Preprocesamiento: RE-BKT**

Este Notebook contiene el flujo de trabajo completo para la preparación, limpieza e identificación de perfiles estudiantiles utilizando los datasets de ASSISTments (2012 y 2017). El objetivo es transformar datos brutos de interacción en un formato estandarizado para el entrenamiento del modelo RE-BKT

#Configuración del Entorno

In [2]:
!pip install optuna seaborn scikit-learn gdown
!pip install kaggle

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import os
import gc

## Obtener Data de ASSISTments 2012

In [ ]:
#Subir el archivo kaggle.json
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle (1).json


In [ ]:
#Descargar el conjunto de datos y descomprimir
!kaggle datasets download -d nicolaswattiez/skillbuilder-data-2009-2010
!unzip skillbuilder-data-2009-2010.zip

Dataset URL: https://www.kaggle.com/datasets/nicolaswattiez/skillbuilder-data-2009-2010
License(s): unknown
skillbuilder-data-2009-2010.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  skillbuilder-data-2009-2010.zip
replace 2012-2013-data-with-predictions-4-final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


##Obtener Data de ASSISTments 2017

In [8]:
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')
path = '/content/drive/My Drive/anonymized_full_release_competition_dataset.csv'

Mounted at /content/drive


##Conexión con GitHub

In [9]:
import os
import sys

USER = "orianapedroza"
REPO = "ExtensionBKT"
BRANCH = "master"

if not os.path.exists(REPO):
    !git clone -b {BRANCH} https://github.com/{USER}/{REPO}.git
else:
    print(f"El repositorio {REPO} ya existe. Actualizando...")
    !cd {REPO} && git pull origin {BRANCH}

ruta_biblioteca = f'/content/{REPO}/sourceRE_BKT'

if ruta_biblioteca not in sys.path:
    sys.path.append(ruta_biblioteca)

print("Entorno configurado: Biblioteca RE-BKT lista para usar.")

El repositorio ExtensionBKT ya existe. Actualizando...
From https://github.com/orianapedroza/ExtensionBKT
 * branch            master     -> FETCH_HEAD
Already up to date.
Entorno configurado: Biblioteca RE-BKT lista para usar.


In [10]:
# Si se requiere usar del drive
import sys
drive.mount('/content/drive')
ruta_proyecto = '/content/drive/MyDrive/RE-BKT/sourceRE-BKT'

if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
from fit import ModelBase
from utils import plot_confusion_matrix
from optimize import optimize_hyperparameters
from data_processing import DataLoader, StudentClustered

#**Preparación de los datos**

In [12]:
# Instanciar herramientas
loader = DataLoader()
clusterer = StudentClustered(n_clusters=2)

In [14]:
# Normalizar y Filtrar datos
#df_raw = loader.clean_2012('2012-2013-data-with-predictions-4-final.csv') #ASSISTmentes 2012
df_raw = loader.clean_2017(path) #ASSISTments 2017

#**Identificación de Perfiles Emocionales**

In [16]:
# Caracterizar estudiantes
print("Extrayendo características por estudiante...")
feature_df = clusterer.extract_features(df_raw)

Extrayendo características por estudiante...


In [17]:
# Aplicar K-Means
print("Aplicando K-Means.")
feature_df, X_scaled = clusterer.run_clustering(feature_df)

Aplicando K-Means.


In [18]:
# Dividir Dataset Final (Train/Test)
print("Dividiendo dataset por usuarios (80% Train, 20% Test)")
train_df, test_df = clusterer.assign_and_split(df_raw, feature_df, test_size=0.2)

print(f"\nProceso completado con éxito")
print(f"Total filas originales: {len(df_raw)}")
print(f"Filas Entrenamiento: {len(train_df)}")
print(f"Filas Prueba: {len(test_df)}")

Dividiendo dataset por usuarios (80% Train, 20% Test)

Proceso completado con éxito
Total filas originales: 942816
Filas Entrenamiento: 744085
Filas Prueba: 198729


##Opcional

In [19]:
# Liberar
del df_raw
del feature_df
gc.collect()

0

In [20]:
# Verificar que no hay estudiantes compartidos entre train y test
train_users = set(train_df['user_id'])
test_users = set(test_df['user_id'])
interseccion = train_users.intersection(test_users)

if len(interseccion) == 0:
    print("\nVerificación de Integridad: No hay fuga de datos.")
else:
    print(f"\nALERTA: Hay {len(interseccion)} usuarios compartidos.")


Verificación de Integridad: No hay fuga de datos.


In [21]:
train_df.to_csv('train_df.csv', index=False)
test_df.to_csv('test_df.csv', index=False)
print("Los archivos train_df.csv y test_df.csv han sido guardados.")

Los archivos train_df.csv y test_df.csv han sido guardados.
